## **각 지자체별 PM 주차구역 전처리/통합**

- 각 지자체별 'OO구 / 주소 / 상세위치 / 거치대유무' 형식의 열 추출 및 정제(제공된 데이터에 한함)

- 처리한 각각의 데이터프레임을 하나의 `merged_df`로 병합

- '서울시 자체 예산 PM 주차 구역 데이터프레임(`seoul_df`)'에 PM 주차 구역 데이터(행) 추가(`merged_df`와 비교) 

- `seoul_df` 데이터 점검, CSV 저장

- 서대문구, 은평구는 제외(수동으로 직접 수정하여 반영)

- **유의사항: 각 자치구별 데이터셋마다 컬럼 형식 등 데이터가 조금씩 다르므로 전처리에 유의**

#### **결과물: 서울시 전동킥보드 주차구역 현황 종합본.csv**

  #### **목표 데이터 설명**
  * **순번** - 첫 번째 행부터 차례대로 부여되는 번호입니다.
  * **시군구명** - PM 주차구역의 자치구 문자열입니다.
  * **주소** - PM 주차구역의 주소입니다. 거의 대부분 '지번 주소'로 이루어져 있습니다.
  * **상세위치** - 주차구역의 상세 위치입니다(예: OO 건물 앞).
  * **거치대유무** - 거치대가 설치되어 있는지 여부입니다(Y/N).

#

In [1]:
# 라이브러리 import
import pandas as pd
import numpy as np

### 서울시 자체 예산 개설 전동킥보드 주차구역 불러오기

#### (이후에 각 지자체별 데이터프레임과 통합)

In [2]:
# 서울시 자체 예산 전동킥보드 주차 구역 데이터셋
seoul_df = pd.read_csv('서울시 전동킥보드 주차구역 현황.csv', encoding='cp949')
seoul_df.rename(columns={'거치대 유무': '거치대유무'}, inplace=True)
print(seoul_df.shape)
display(seoul_df.head(3))

# 열별 데이터타입 확인
print("\n열별 데이터타입:")
print(seoul_df.dtypes)

# 열별 결측치 개수 확인
print("\n열별 결측치 개수:")
print(seoul_df.isnull().sum())

# 열별 빈 문자열 개수 확인
print("\n열별 빈 문자열 개수:")
print((seoul_df == '').sum())

(329, 5)


,순번,시군구명,주소,상세위치,거치대유무
0,1,종로구,팔판동 115-63,청와대 춘추문 맞은편,Y
1,2,종로구,연건동 218-1,KT 광화문 혜화지사 앞,Y
2,3,종로구,연건동 178-3,홍익대학교 대학로 맞은편,Y



열별 데이터타입:
순번       int64
시군구명       str
주소         str
상세위치       str
거치대유무      str
dtype: object

열별 결측치 개수:
순번       0
시군구명     0
주소       0
상세위치     0
거치대유무    0
dtype: int64

열별 빈 문자열 개수:
순번       0
시군구명     0
주소       0
상세위치     0
거치대유무    0
dtype: int64


#

### 데이터셋 정제 및 전처리를 위한 함수 정의

#### 필요 컬럼 및 데이터 추출하여 반환
`check_columns = ['구', '도로명주소', '지번주소']`

In [3]:
# 필요 컬럼 리스트
required_columns = ['구', '도로명주소', '지번주소', '상세위치', '거치대유무']


# 주소 정제 및 데이터 처리 함수
def preprocess_data(df, city_name="서울특별시", district_name=None):
    # 필요한 컬럼이 존재하는지 확인
    check_columns = ['구', '도로명주소', '지번주소']
    for col in check_columns:
        if col not in df.columns:
            raise ValueError(f"'{col}' 컬럼이 데이터프레임에 존재하지 않습니다.")

    # 시군구명의 경우, 서울특별시 문자열 지우고 양옆 공백 제거
    df['구'] = df['구'].str.replace(city_name, '').str.strip()

    # 도로명 주소, 지번 주소는 '서울특별시 OO구' 문자열을 지우고 양 옆 공백 제거
    # 만약 district_name이 제공된다면, '서울특별시 OO구' 형태로 제거
    if district_name:
        full_district_name = f"{city_name} {district_name}"
        df['도로명주소'] = df['도로명주소'].str.replace(full_district_name, '').str.strip()
        df['지번주소'] = df['지번주소'].str.replace(full_district_name, '').str.strip()
        # 추가로, '서울특별시' 문자열도 제거
        df['도로명주소'] = df['도로명주소'].str.replace(city_name, '').str.strip()
        df['지번주소'] = df['지번주소'].str.replace(city_name, '').str.strip()
    return df


# 컬럼 정보가 'O', 'X'로 되어 있는 경우, 'O'는 'Y'로, 'X'는 'N'으로 변환하는 함수
def convert_ox_to_ny(df, column_name):
    if column_name in df.columns:
        df[column_name] = df[column_name].str.lower().str.strip()
        df[column_name] = df[column_name].map({'o': 'Y', 'x': 'N'})
    else:
        raise ValueError(f"'{column_name}' 컬럼이 데이터프레임에 존재하지 않습니다.")
    return df


# 데이터셋 정보 확인
def print_basic_info(df, name="데이터프레임"):
    print(f"{name}의 형태: {df.shape}")
    print(f"{name}의 컬럼명: {df.columns.tolist()}")
    print(f"{name}의 상위 3개 행:")
    display(df.head(3))

#

#### 강남구 데이터 추출 및 정제

In [4]:
# 강남구 데이터셋 불러오기
gangnam_df = pd.read_csv('서울특별시 강남구_전동킥보드 주차구역_20240319.csv', encoding='cp949')
print_basic_info(gangnam_df, "강남구 데이터셋")

강남구 데이터셋의 형태: (10, 4)
강남구 데이터셋의 컬럼명: ['시군구명', '도로명 주소', '지번 주소', '세부위치']
강남구 데이터셋의 상위 3개 행:


,시군구명,도로명 주소,지번 주소,세부위치
0,서울특별시 강남구,서울특별시 강남구 학동로 401,서울특별시 강남구 청담동 41-2,강남구청역 4번 출구
1,서울특별시 강남구,서울특별시 강남구 봉은사로 502,서울특별시 강남구 삼성동 145,삼성중앙역 5번 출구
2,서울특별시 강남구,서울특별시 강남구 테헤란로 202,서울특별시 강남구 역삼동 717,역삼역 8번 출구


In [5]:
gangnam_df.rename(columns={'시군구명': '구',
                            '도로명 주소': '도로명주소', 
                            '지번 주소': '지번주소',
                            '세부위치': '상세위치'
                            }, inplace=True)
gangnam_df['거치대유무'] = np.nan  # 거치대유무 컬럼 추가 (값은 NaN으로 초기화)

gangnam_df = preprocess_data(gangnam_df, city_name="서울특별시", district_name="강남구")

print_basic_info(gangnam_df, "전처리된 강남구 데이터셋")

전처리된 강남구 데이터셋의 형태: (10, 5)
전처리된 강남구 데이터셋의 컬럼명: ['구', '도로명주소', '지번주소', '상세위치', '거치대유무']
전처리된 강남구 데이터셋의 상위 3개 행:


,구,도로명주소,지번주소,상세위치,거치대유무
0,강남구,학동로 401,청담동 41-2,강남구청역 4번 출구,NaN
1,강남구,봉은사로 502,삼성동 145,삼성중앙역 5번 출구,NaN
2,강남구,테헤란로 202,역삼동 717,역삼역 8번 출구,NaN


#

#### 강북구 데이터 추출 및 정제

In [6]:
gangbuk_df = pd.read_csv('서울특별시 강북구_킥보드 주차구역 위치_20251104.csv', encoding='cp949')
print_basic_info(gangbuk_df, "강북구 데이터셋")

강북구 데이터셋의 형태: (9, 7)
강북구 데이터셋의 컬럼명: ['연번', '시군구명', '지번주소', '세부위치', '거치대 유무', '설치일자', '기준일자']
강북구 데이터셋의 상위 3개 행:


,연번,시군구명,지번주소,세부위치,거치대 유무,설치일자,기준일자
0,1,서울특별시 강북구,서울특별시 강북구 도봉로 317,수유역 1번출구,x,2022-07-29,2024-11-04
1,2,서울특별시 강북구,서울특별시 강북구 미아동 42-11,다이소 미아사거리점 앞,x,2022-12-09,2024-11-04
2,3,서울특별시 강북구,서울특별시 강북구 미아동 125-33,미아역 4번 출구 앞,x,2022-12-09,2024-11-04


In [7]:
# 동일 방식 진행
gangbuk_df['도로명주소'] = ''  # 도로명주소 컬럼이 없으므로, 빈 컬럼 생성
gangbuk_df.rename(columns={'시군구명': '구',
                            '지번 주소': '지번주소', 
                            '거치대 유무': '거치대유무', 
                            '세부위치': '상세위치'
                            }, inplace=True)
gangbuk_df = preprocess_data(gangbuk_df, city_name="서울특별시", district_name="강북구")

# 필요한 컬럼만 선택
gangbuk_df = gangbuk_df[required_columns]

# 거치대유무 컬럼의 값이 'O', 'X'로 되어 있을 경우, 'Y', 'N'으로 변환
gangbuk_df = convert_ox_to_ny(gangbuk_df, '거치대유무')

print_basic_info(gangbuk_df, "전처리된 강북구 데이터셋")

전처리된 강북구 데이터셋의 형태: (9, 5)
전처리된 강북구 데이터셋의 컬럼명: ['구', '도로명주소', '지번주소', '상세위치', '거치대유무']
전처리된 강북구 데이터셋의 상위 3개 행:


,구,도로명주소,지번주소,상세위치,거치대유무
0,강북구,,도봉로 317,수유역 1번출구,N
1,강북구,,미아동 42-11,다이소 미아사거리점 앞,N
2,강북구,,미아동 125-33,미아역 4번 출구 앞,N


#

#### 강서구 데이터 추출 및 정제

In [8]:
gangseo_df = pd.read_csv('서울특별시 강서구_전동킥보드 주차구역_20250801.csv', encoding='cp949')
print_basic_info(gangseo_df, "강서구 데이터셋")

강서구 데이터셋의 형태: (34, 4)
강서구 데이터셋의 컬럼명: ['연번', '전동킥보드 주차구역', '법정동', '데이터기준일자']
강서구 데이터셋의 상위 3개 행:


,연번,전동킥보드 주차구역,법정동,데이터기준일자
0,1,발산역 8번 출구(마곡동 727-1222),마곡동,2025-08-01
1,2,마곡역 3번출구(마곡동 727-1428),마곡동,2025-08-01
2,3,마곡역 6번 출구(마곡동 728-163),마곡동,2025-08-01


In [9]:
gangseo_df['구'] = '강서구'         # 구 컬럼이 없으므로, '강서구'로 채워진 구 컬럼 생성
gangseo_df['도로명주소'] = ''       # 도로명주소 컬럼이 없으므로, 빈 컬럼 생성
gangseo_df['거치대유무'] = np.nan   # 거치대유무 컬럼 추가 (값은 NaN으로 초기화)
gangseo_df.rename(columns={'전동킥보드 주차구역': '지번주소'}, inplace=True)

# 지번주소에서 상세 주소(괄호 전까지)만 추출
gangseo_df['상세위치'] = gangseo_df['지번주소'].str.split('(').str[0].str.strip()
# 지번주소에서 괄호 안의 주소만 추출
gangseo_df['지번주소'] = gangseo_df['지번주소'].str.split('(').str[1].str.rstrip(')')

gangseo_df = preprocess_data(gangseo_df, city_name="서울특별시", district_name="강서구")

# 필요한 컬럼만 선택
gangseo_df = gangseo_df[required_columns]

print_basic_info(gangseo_df, "전처리된 강서구 데이터셋")

전처리된 강서구 데이터셋의 형태: (34, 5)
전처리된 강서구 데이터셋의 컬럼명: ['구', '도로명주소', '지번주소', '상세위치', '거치대유무']
전처리된 강서구 데이터셋의 상위 3개 행:


,구,도로명주소,지번주소,상세위치,거치대유무
0,강서구,,마곡동 727-1222,발산역 8번 출구,NaN
1,강서구,,마곡동 727-1428,마곡역 3번출구,NaN
2,강서구,,마곡동 728-163,마곡역 6번 출구,NaN


#

#### 동대문구 데이터 추출 및 정제

In [10]:
dongdaemun_df = pd.read_csv('서울특별시 동대문구_전동킥보드 주차구역_20260319.csv', encoding='cp949')
print_basic_info(dongdaemun_df, "동대문구 데이터셋")

동대문구 데이터셋의 형태: (13, 6)
동대문구 데이터셋의 컬럼명: ['연번', '시군구명', '주소', '세부위치', '거치대 유무', '데이터기준일자']
동대문구 데이터셋의 상위 3개 행:


,연번,시군구명,주소,세부위치,거치대 유무,데이터기준일자
0,1,서울특별시 동대문구,장안동 416-10,창도빌딩 앞(지하철 환풍구 옆 가로등 가로수 사이),X,2026-03-19
1,2,서울특별시 동대문구,답십리동 497-23,천호대로 257 청계푸르지오시티오피스텔 앞,X,2026-03-19
2,3,서울특별시 동대문구,신설동 98-32,신설동역 5번출구 뒤 자전거보관대 옆(파리바게트 앞),X,2026-03-19


In [11]:
dongdaemun_df['도로명주소'] = ''  # 도로명주소 컬럼이 없으므로, 빈 컬럼 생성
dongdaemun_df.rename(columns={'시군구명': '구',
                            '주소': '지번주소',
                            '세부위치': '상세위치',
                            '거치대 유무': '거치대유무' 
                            }, inplace=True)

dongdaemun_df = convert_ox_to_ny(dongdaemun_df, '거치대유무')
dongdaemun_df = preprocess_data(dongdaemun_df, city_name="서울특별시", district_name="동대문구")

# 필요한 컬럼만 선택
dongdaemun_df = dongdaemun_df[required_columns]

print_basic_info(dongdaemun_df, "전처리된 동대문구 데이터셋")

전처리된 동대문구 데이터셋의 형태: (13, 5)
전처리된 동대문구 데이터셋의 컬럼명: ['구', '도로명주소', '지번주소', '상세위치', '거치대유무']
전처리된 동대문구 데이터셋의 상위 3개 행:


,구,도로명주소,지번주소,상세위치,거치대유무
0,동대문구,,장안동 416-10,창도빌딩 앞(지하철 환풍구 옆 가로등 가로수 사이),N
1,동대문구,,답십리동 497-23,천호대로 257 청계푸르지오시티오피스텔 앞,N
2,동대문구,,신설동 98-32,신설동역 5번출구 뒤 자전거보관대 옆(파리바게트 앞),N


#

#### 동작구 데이터 추출 및 정제

In [12]:
dongjak_df = pd.read_csv('서울특별시 동작구_공유형 전동킥보드 주차구역_20260316.csv', encoding='cp949')
print_basic_info(dongjak_df, "동작구 데이터셋")

동작구 데이터셋의 형태: (3, 6)
동작구 데이터셋의 컬럼명: ['연번', '시군구', '상세주소', '부가설명', '설치일자', '데이터기준일자']
동작구 데이터셋의 상위 3개 행:


,연번,시군구,상세주소,부가설명,설치일자,데이터기준일자
0,1,서울특별시 동작구,본동 344-10,노들역 1번 출구 측면,2022-07-27,2026-03-16
1,2,서울특별시 동작구,대방동 356-2,공군호텔 버스정류소 인근,2022-07-27,2026-03-16
2,3,서울특별시 동작구,상도동 511,숭실대입구 3번출구,2023-12-22,2026-03-16


In [13]:
dongjak_df['도로명주소'] = ''  # 도로명주소 컬럼이 없으므로, 빈 컬럼 생성
dongjak_df['거치대유무'] = np.nan  # 거치대유무 컬럼 추가 (값은 NaN으로 초기화)
dongjak_df.rename(columns={'시군구': '구', 
                           '상세주소': '지번주소', 
                           '부가설명': '상세위치'
                           }, inplace=True)

dongjak_df = preprocess_data(dongjak_df, city_name="서울특별시", district_name="동작구")

# 필요한 컬럼만 선택
dongjak_df = dongjak_df[required_columns]

print_basic_info(dongjak_df, "전처리된 동작구 데이터셋")

전처리된 동작구 데이터셋의 형태: (3, 5)
전처리된 동작구 데이터셋의 컬럼명: ['구', '도로명주소', '지번주소', '상세위치', '거치대유무']
전처리된 동작구 데이터셋의 상위 3개 행:


,구,도로명주소,지번주소,상세위치,거치대유무
0,동작구,,본동 344-10,노들역 1번 출구 측면,NaN
1,동작구,,대방동 356-2,공군호텔 버스정류소 인근,NaN
2,동작구,,상도동 511,숭실대입구 3번출구,NaN


#

### 서초구 데이터 추출 및 정제

In [14]:
seocho_df = pd.read_csv('서울특별시 서초구_전동킥보드 주차구역 현황_20250212.csv', encoding='cp949')
print_basic_info(seocho_df, "서초구 데이터셋")

서초구 데이터셋의 형태: (48, 3)
서초구 데이터셋의 컬럼명: ['주소', '세부위치', '설치년월']
서초구 데이터셋의 상위 3개 행:


,주소,세부위치,설치년월
0,서초동 1374,대륭서초타워 앞(우성아파트 앞 사거리),2022-12-05
1,서초동 1374,현대렉시온 앞(뱅뱅사거리 근처),2022-12-05
2,서초동 1375,강남대로 올리브영 앞,2022-12-05


In [15]:
seocho_df['구'] = '서초구'          # 구 컬럼이 없으므로, '서초구'로 채워진 구 컬럼 생성
seocho_df['도로명주소'] = ''        # 도로명주소 컬럼이 없으므로, 빈 컬럼 생성
seocho_df['거치대유무'] = np.nan    # 거치대유무 컬럼 추가 (값은 NaN으로 초기화)
seocho_df.rename(columns={'주소': '지번주소',
                          '세부위치': '상세위치'
                          }, inplace=True)

seocho_df = preprocess_data(seocho_df, city_name="서울특별시", district_name="서초구")

# 필요한 컬럼만 선택
seocho_df = seocho_df[required_columns]

print_basic_info(seocho_df, "전처리된 서초구 데이터셋")

전처리된 서초구 데이터셋의 형태: (48, 5)
전처리된 서초구 데이터셋의 컬럼명: ['구', '도로명주소', '지번주소', '상세위치', '거치대유무']
전처리된 서초구 데이터셋의 상위 3개 행:


,구,도로명주소,지번주소,상세위치,거치대유무
0,서초구,,서초동 1374,대륭서초타워 앞(우성아파트 앞 사거리),NaN
1,서초구,,서초동 1374,현대렉시온 앞(뱅뱅사거리 근처),NaN
2,서초구,,서초동 1375,강남대로 올리브영 앞,NaN


#

#### 성북구 데이터 추출 및 정제

In [16]:
seongbuk_df = pd.read_csv('서울특별시 성북구_전동 킥보드 주차구역 위치_20230126.csv', encoding='cp949')
print_basic_info(seongbuk_df, "성북구 데이터셋")

성북구 데이터셋의 형태: (3, 8)
성북구 데이터셋의 컬럼명: ['데이터기준일', '연번', '시도명', '시군구명', '주차장설치장소', '주차대수', '설치(준공)연도', '관리기관']
성북구 데이터셋의 상위 3개 행:


,데이터기준일,연번,시도명,시군구명,주차장설치장소,주차대수,설치(준공)연도,관리기관
0,2023-01-26,1,서울특별시,성북구,석관동 375-54(석계역 7번 출구),6,2022-12-19,교통행정과
1,2023-01-26,2,서울특별시,성북구,월곡동 37-4(월곡역 3번 출구),6,2022-12-19,교통행정과
2,2023-01-26,3,서울특별시,성북구,종암동 3-1288(종암박스파크),8,2022-12-19,교통행정과


In [17]:
seongbuk_df['도로명주소'] = ''  # 도로명주소 컬럼이 없으므로, 빈 컬럼 생성
seongbuk_df['거치대유무'] = np.nan  # 거치대유무 컬럼 추가 (값은 NaN으로 초기화)
seongbuk_df.rename(columns={'시군구명': '구',
                            '주차장설치장소': '지번주소', 
                            }, inplace=True)

# 지번주소에서 상세 주소(괄호 안)만 추출
seongbuk_df['상세위치'] = seongbuk_df['지번주소'].str.split('(').str[1].str.rstrip(')')
# 지번주소에서 괄호 전까지의 주소 추출
seongbuk_df['지번주소'] = seongbuk_df['지번주소'].str.split('(').str[0].str.strip()

seongbuk_df = preprocess_data(seongbuk_df, city_name="서울특별시", district_name="성북구")

# 필요한 컬럼만 선택
seongbuk_df = seongbuk_df[required_columns]

print_basic_info(seongbuk_df, "전처리된 성북구 데이터셋")

전처리된 성북구 데이터셋의 형태: (3, 5)
전처리된 성북구 데이터셋의 컬럼명: ['구', '도로명주소', '지번주소', '상세위치', '거치대유무']
전처리된 성북구 데이터셋의 상위 3개 행:


,구,도로명주소,지번주소,상세위치,거치대유무
0,성북구,,석관동 375-54,석계역 7번 출구,NaN
1,성북구,,월곡동 37-4,월곡역 3번 출구,NaN
2,성북구,,종암동 3-1288,종암박스파크,NaN


#

#### 용산구 데이터 추출 및 정제

In [18]:
yongsan_df = pd.read_csv('서울특별시 용산구_전동킥보드 주차 구역_20250805.csv', encoding='cp949')
print_basic_info(yongsan_df, "용산구 데이터셋")

용산구 데이터셋의 형태: (5, 7)
용산구 데이터셋의 컬럼명: ['연번', '행정동', '설치장소', '상세주소', '설치형태', '준공일자', '데이터기준일']
용산구 데이터셋의 상위 3개 행:


,연번,행정동,설치장소,상세주소,설치형태,준공일자,데이터기준일
0,1.0,남영동,숙대입구역 10번출구 자전거보관소 옆,서울특별시 용산구 갈월동 69-27,노면표시 및 교통안전표지,2022-07-25,2023-08-21
1,2.0,용문동,효창공원앞역 5번출구,서울특별시 용산구 용문동 5-157,거치대+노면표시+주차구역 표지,2022-12-12,2023-08-21
2,3.0,이태원1동,스타벅스 이태원역점 옆,서울특별시 용산구 이태원동 127-6,거치대+노면표시+주차구역 표지,2022-12-12,2023-08-21


In [19]:
yongsan_df['구'] = '용산구'          # 구 컬럼이 없으므로, '용산구'로 채워진 구 컬럼 생성
yongsan_df['도로명주소'] = ''        # 도로명주소 컬럼이 없으므로, 빈 컬럼 생성
yongsan_df['거치대유무'] = np.nan    # 거치대유무 컬럼 추가 (값은 NaN으로 초기화)
yongsan_df.rename(columns={'상세주소': '지번주소',
                          '설치장소': '상세위치'
                          }, inplace=True)

yongsan_df = preprocess_data(yongsan_df, city_name="서울특별시", district_name="용산구")

# 필요한 컬럼만 선택
yongsan_df = yongsan_df[required_columns]

print_basic_info(yongsan_df, "전처리된 용산구 데이터셋")

전처리된 용산구 데이터셋의 형태: (5, 5)
전처리된 용산구 데이터셋의 컬럼명: ['구', '도로명주소', '지번주소', '상세위치', '거치대유무']
전처리된 용산구 데이터셋의 상위 3개 행:


,구,도로명주소,지번주소,상세위치,거치대유무
0,용산구,,갈월동 69-27,숙대입구역 10번출구 자전거보관소 옆,NaN
1,용산구,,용문동 5-157,효창공원앞역 5번출구,NaN
2,용산구,,이태원동 127-6,스타벅스 이태원역점 옆,NaN


#

#### 마포구 데이터 추출 및 정제

In [20]:
mapo_df = pd.read_csv('서울특별시_마포구_전동킥보드 주차 위치_20230511.csv', encoding='cp949')
print_basic_info(mapo_df, "마포구 데이터셋")

마포구 데이터셋의 형태: (21, 5)
마포구 데이터셋의 컬럼명: ['연번', '위치', '행정동', '설치연도', 'Unnamed: 4']
마포구 데이터셋의 상위 3개 행:


,연번,위치,행정동,설치연도,Unnamed: 4
0,1.0,대흥역 1번 출구,서울특별시 마포구 대흥동 216-4,2022.0,NaN
1,2.0,마포구청역 3번 출구,서울특별시 마포구 성산동 592-6,2022.0,NaN
2,3.0,아현역 4번 출구,서울특별시 마포구 아현동 329-15,2022.0,NaN


In [21]:
mapo_df['구'] = '마포구'          # 구 컬럼이 없으므로, '마포구'로 채워진 구 컬럼 생성
mapo_df['도로명주소'] = ''        # 도로명주소 컬럼이 없으므로, 빈 컬럼 생성
mapo_df['거치대유무'] = np.nan    # 거치대유무 컬럼 추가 (값은 NaN으로 초기화)
mapo_df.rename(columns={'행정동': '지번주소', 
                        '위치': '상세위치'
                        }, inplace=True)

mapo_df = preprocess_data(mapo_df, city_name="서울특별시", district_name="마포구")

# 필요한 컬럼만 선택
mapo_df = mapo_df[required_columns]

print_basic_info(mapo_df, "전처리된 마포구 데이터셋")

전처리된 마포구 데이터셋의 형태: (21, 5)
전처리된 마포구 데이터셋의 컬럼명: ['구', '도로명주소', '지번주소', '상세위치', '거치대유무']
전처리된 마포구 데이터셋의 상위 3개 행:


,구,도로명주소,지번주소,상세위치,거치대유무
0,마포구,,대흥동 216-4,대흥역 1번 출구,NaN
1,마포구,,성산동 592-6,마포구청역 3번 출구,NaN
2,마포구,,아현동 329-15,아현역 4번 출구,NaN


#

### **추출/정제한 지자체별 PM 주차 구역 데이터프레임 병합**
- 각 지자체별에서 추출한 `DataFrame`들을 하나의 데이터프레임으로 병합
- `pd.merge()`의 `indicator` 옵션을 사용하여 도로명주소와 지번주소가 기존 `seoul_df`에 존재하는지 판별
- `Numpy`와 필터링(불리언 인덱싱)을 통해 완전히 새로운 PM 주차구역 데이터만 추가

In [22]:
# 추출/정제한 지자체별 PM 주차 구역 데이터프레임 병합
# 서대문구, 은평구는 제외
merged_df = pd.concat([
    gangnam_df, 
    gangbuk_df, 
    gangseo_df, 
    dongdaemun_df, 
    dongjak_df, 
    seocho_df, 
    seongbuk_df, 
    yongsan_df, 
    mapo_df
], ignore_index=True)

print_basic_info(merged_df, "병합된 데이터셋")
print("병합된 데이터셋의 하위 3개 행:")
display(merged_df.tail(3))

# 병합된 데이터셋의 열별 데이터타입 확인
print("\n병합된 데이터셋의 열별 데이터타입:")
print(merged_df.dtypes)
# 병합된 데이터셋의 열별 결측치 개수 확인
print("\n병합된 데이터셋의 열별 결측치 개수:")
print(merged_df.isnull().sum())
# 병합된 데이터셋의 열별 빈 문자열 개수 확인
print("\n병합된 데이터셋의 열별 빈 문자열 개수:")
print((merged_df == '').sum())

print('-' * 100 + '\n')
print_basic_info(seoul_df, "서울시 자체 예산 데이터셋")

병합된 데이터셋의 형태: (146, 5)
병합된 데이터셋의 컬럼명: ['구', '도로명주소', '지번주소', '상세위치', '거치대유무']
병합된 데이터셋의 상위 3개 행:


,구,도로명주소,지번주소,상세위치,거치대유무
0,강남구,학동로 401,청담동 41-2,강남구청역 4번 출구,NaN
1,강남구,봉은사로 502,삼성동 145,삼성중앙역 5번 출구,NaN
2,강남구,테헤란로 202,역삼동 717,역삼역 8번 출구,NaN


병합된 데이터셋의 하위 3개 행:


,구,도로명주소,지번주소,상세위치,거치대유무
143,마포구,,동교동,홍대입구역 3번,NaN
144,마포구,,동교동,홍대입구역 2번,NaN
145,마포구,,동교동,홍대입구역 1번,NaN



병합된 데이터셋의 열별 데이터타입:
구           str
도로명주소       str
지번주소     object
상세위치     object
거치대유무    object
dtype: object

병합된 데이터셋의 열별 결측치 개수:
구          0
도로명주소      0
지번주소       1
상세위치       1
거치대유무    124
dtype: int64

병합된 데이터셋의 열별 빈 문자열 개수:
구          0
도로명주소    136
지번주소       0
상세위치       0
거치대유무      0
dtype: int64
----------------------------------------------------------------------------------------------------

서울시 자체 예산 데이터셋의 형태: (329, 5)
서울시 자체 예산 데이터셋의 컬럼명: ['순번', '시군구명', '주소', '상세위치', '거치대유무']
서울시 자체 예산 데이터셋의 상위 3개 행:


,순번,시군구명,주소,상세위치,거치대유무
0,1,종로구,팔판동 115-63,청와대 춘추문 맞은편,Y
1,2,종로구,연건동 218-1,KT 광화문 혜화지사 앞,Y
2,3,종로구,연건동 178-3,홍익대학교 대학로 맞은편,Y


#### 병합 DF 전처리

In [23]:
# merged_df의 열별 데이터타입을 seoul_df의 열별 데이터타입과 맞추기
merged_df['지번주소'] = merged_df['지번주소'].astype(str)
merged_df['상세위치'] = merged_df['상세위치'].astype(str)
merged_df['거치대유무'] = merged_df['거치대유무'].astype(str)

# '지번주소', '상세위치' 열의 결측치(NaN)를 빈 문자열('')로 대체하여 문자열 타입으로 변환
merged_df['지번주소'] = merged_df['지번주소'].fillna('')
merged_df['상세위치'] = merged_df['상세위치'].fillna('')

# 거치대유무가 NaN인 경우, 'N'으로 채우기 (서울시 데이터셋에서는 거치대가 없는 경우 'N'으로 표기되어 있기 때문)
merged_df['거치대유무'] = merged_df['거치대유무'].fillna('N')

# merged_df에서 도로명주소와 지번주소가 모두 빈 문자열인 행 제거 (비어있는 주소는 매칭이 불가능하므로 제거)
merged_df = merged_df[(merged_df['도로명주소'] != '') | (merged_df['지번주소'] != '')]

# seoul_df에서 비교할 기준 주소 목록 만들기 (공백 제거 및 고유값 추출)
# 빈 문자열이나 NaN 값이 잘못 매칭되는 것을 방지하기 위해 정제된 주소 목록 생성(seoul_addrs)
seoul_addrs = seoul_df['주소'].astype(str).str.strip()
seoul_addrs = seoul_addrs[seoul_addrs != ''].unique()   # 비어있지 않은 주소만 추출

# merged_df의 주소 데이터 전처리 (비교를 위해 공백 제거)
# 결측치(NaN)가 문자열 'nan'으로 바뀌는 것을 방지하기 위해 빈 문자열('')로 변경
merged_df['도로명주소'] = merged_df['도로명주소'].fillna('').astype(str).str.strip().replace('nan', '')
merged_df['지번주소'] = merged_df['지번주소'].fillna('').astype(str).str.strip().replace('nan', '')

#### 비교 및 검사 수행

In [24]:
# 필터링(boolean indexing) 조건 만들기
# 조건: (도로명주소가 seoul_addrs에 없고) AND (지번주소가 seoul_addrs에 없음)
# 빈 문자열('')은 seoul_addrs에 포함되어 있지 않으므로 자연스럽게 무시됨
condition = (~merged_df['도로명주소'].isin(seoul_addrs)) & (~merged_df['지번주소'].isin(seoul_addrs))

# 조건에 맞는 새로운 데이터만 추출
new_df = merged_df[condition].copy()


# 추출된 데이터를 seoul_df 양식에 맞게 맞추기
# np.where: 지번주소가 있으면 지번주소를, 없으면 도로명주소를 '주소' 컬럼으로 지정
new_df['주소'] = np.where(
    new_df['지번주소'] != '', 
    new_df['지번주소'], 
    new_df['도로명주소']
)

# 컬럼명 변경 및 seoul_df 순서에 맞게 재배치
new_df = new_df.rename(columns={'구': '시군구명'})
new_df = new_df[['시군구명', '주소', '상세위치', '거치대유무']]

# 시군구명, 주소를 기준으로 오름차순 정렬
new_df = new_df.sort_values(by=['시군구명', '주소'], ascending=[True, True]).reset_index(drop=True)

# 기존 seoul_df와 병합 및 순번 재정렬
final_df = pd.concat([seoul_df, new_df], ignore_index=True)
final_df['순번'] = range(1, len(final_df) + 1)

print(f"기존 서울시 데이터: {len(seoul_df)}개")
print(f"새롭게 추가된 데이터: {len(new_df)}개")
print(f"최종 병합된 데이터: {len(final_df)}개")

final_df

기존 서울시 데이터: 329개
새롭게 추가된 데이터: 66개
최종 병합된 데이터: 395개


,순번,시군구명,주소,상세위치,거치대유무
0,1,종로구,팔판동 115-63,청와대 춘추문 맞은편,Y
1,2,종로구,연건동 218-1,KT 광화문 혜화지사 앞,Y
2,3,종로구,연건동 178-3,홍익대학교 대학로 맞은편,Y
3,4,종로구,동승동 1-24,대학로 마로니에 공원 앞,Y
4,5,종로구,와룡동 75-4,연악사 맞은편,Y
...,...,...,...,...,...
390,391,서초구,양재동 67-10,교육개발원 입구 사거리(1),N
391,392,성북구,석관동 375-54,석계역 7번 출구,N
392,393,성북구,월곡동 37-4,월곡역 3번 출구,N
393,394,성북구,종암동 3-1288,종암박스파크,N


### 데이터셋 점검 및 저장

In [25]:
print(f"최종 데이터셋 크기: {final_df.shape[0]}개 행, {final_df.shape[1]}개 열\n")

# final_df: 각 열별 데이터 타입 확인
print("각 열별 데이터 타입:")
print(final_df.dtypes)
print('-' * 75)

# 각 열별 결측치 개수 확인
print("각 열별 결측치 개수:")
print(final_df.isnull().sum())
print('-' * 75)
# 결측치 있는 행 출력
print("결측치 있는 행:")
display(final_df[final_df.isnull().any(axis=1)])
print('-' * 75)

# 각 열별 빈 문자열 있는지 확인
print("각 열별 빈 문자열 개수:")
print((final_df == '').sum())
print('-' * 75)
# 빈 문자열 있는 행 출력
print("빈 문자열 있는 행:")
display(final_df[(final_df == '').any(axis=1)])
print('-' * 75)



최종 데이터셋 크기: 395개 행, 5개 열

각 열별 데이터 타입:
순번       int64
시군구명       str
주소         str
상세위치       str
거치대유무      str
dtype: object
---------------------------------------------------------------------------
각 열별 결측치 개수:
순번       0
시군구명     0
주소       0
상세위치     0
거치대유무    0
dtype: int64
---------------------------------------------------------------------------
결측치 있는 행:


,순번,시군구명,주소,상세위치,거치대유무


---------------------------------------------------------------------------
각 열별 빈 문자열 개수:
순번       0
시군구명     0
주소       0
상세위치     0
거치대유무    0
dtype: int64
---------------------------------------------------------------------------
빈 문자열 있는 행:


,순번,시군구명,주소,상세위치,거치대유무


---------------------------------------------------------------------------


In [26]:
# 최종 데이터셋을 CSV 파일로 저장
final_df.to_csv('서울시 전동킥보드 주차구역 현황 종합본.csv', index=False, encoding='utf-8-sig')
print("최종 데이터셋이 '서울시 전동킥보드 주차구역 현황 종합본.csv' 파일로 저장되었습니다.")

최종 데이터셋이 '서울시 전동킥보드 주차구역 현황 종합본.csv' 파일로 저장되었습니다.


In [27]:
temp = pd.read_csv('서울시 전동킥보드 주차구역 현황 종합본.csv', encoding='utf-8-sig')
temp.info()

<class 'pandas.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   순번      395 non-null    int64
 1   시군구명    395 non-null    str  
 2   주소      395 non-null    str  
 3   상세위치    395 non-null    str  
 4   거치대유무   395 non-null    str  
dtypes: int64(1), str(4)
memory usage: 15.6 KB
